In [2]:
!pip install snntorch torch torchvision opencv-python

In [3]:
import os
import glob
import json
import random
import zipfile
import time
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

import cv2
import numpy as np
import torch
import torch.nn as nn
import snntorch as snn
from snntorch import surrogate

# ==============================================================================
# 1. TIỀN XỬ LÝ & NẠP TOÀN BỘ DỮ LIỆU CROP VÀO RAM (~200MB)
# ==============================================================================
def letterbox_gray(img, target_size=64):
    """Resize giữ nguyên tỷ lệ khung hình bằng cách thêm viền đen (Letterbox)"""
    h, w = img.shape
    scale = target_size / max(h, w)
    nh, nw = max(1, int(h * scale)), max(1, int(w * scale))
    resized = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)
    canvas = np.zeros((target_size, target_size), dtype=np.uint8)
    top, left = (target_size - nh) // 2, (target_size - nw) // 2
    canvas[top:top+nh, left:left+nw] = resized
    return canvas

def preload_crops_to_tensors(json_path, image_path_map, img_size=64, min_box_size=20, max_per_class=5000):
    """Đọc mỗi ảnh gốc đúng 1 lần, cắt toàn bộ đối tượng và trả về Tensor uint8"""
    t0 = time.time()
    with open(json_path, 'r') as f:
        coco_data = json.load(f)

    id_to_fname = {img['id']: os.path.basename(img['file_name']) for img in coco_data['images']}
    cat_ids = sorted([cat['id'] for cat in coco_data['categories']])
    cat_map = {cid: idx for idx, cid in enumerate(cat_ids)}

    by_class = defaultdict(list)
    for ann in coco_data['annotations']:
        fname = id_to_fname.get(ann['image_id'])
        if fname in image_path_map:
            x, y, w, h = ann['bbox']
            if w >= min_box_size and h >= min_box_size:
                cid = cat_map[ann['category_id']]
                by_class[cid].append((image_path_map[fname], int(x), int(y), int(w), int(h), cid))

    random.seed(42)
    selected_by_img = defaultdict(list)
    for cid, items in by_class.items():
        if len(items) > max_per_class:
            items = random.sample(items, max_per_class)
        for path, x, y, w, h, label in items:
            selected_by_img[path].append((x, y, w, h, label))

    def process_single_image(item):
        path, boxes = item
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            return [], []
        H, W = img.shape
        crops, labels = [], []
        for (x, y, w, h, label) in boxes:
            # Mở rộng 15% vùng bao quanh để lấy ngữ cảnh mặt đường/bóng đổ
            pad_w, pad_h = int(w * 0.15), int(h * 0.15)
            x1, y1 = max(0, x - pad_w), max(0, y - pad_h)
            x2, y2 = min(W, x + w + pad_w), min(H, y + h + pad_h)
            crop = img[y1:y2, x1:x2]
            if crop.size > 0:
                crops.append(letterbox_gray(crop, img_size))
                labels.append(label)
        return crops, labels

    all_x, all_y = [], []
    with ThreadPoolExecutor(max_workers=8) as ex:
        for crops, labels in ex.map(process_single_image, selected_by_img.items()):
            all_x.extend(crops)
            all_y.extend(labels)

    X_tensor = torch.from_numpy(np.array(all_x, dtype=np.uint8)).unsqueeze(1)  # [N, 1, 64, 64]
    Y_tensor = torch.tensor(all_y, dtype=torch.long)
    print(f"[OK] Nạp {len(Y_tensor)} mẫu trong {time.time()-t0:.1f}s ({X_tensor.nbytes / 1e6:.1f} MB)")
    return X_tensor, Y_tensor

# Quét đường dẫn toàn bộ ảnh trong /kaggle/input
INPUT_DIR = "/kaggle/input"
all_files = glob.glob(os.path.join(INPUT_DIR, "**/*"), recursive=True)
image_path_map = {
    os.path.basename(f): f 
    for f in all_files 
    if os.path.isfile(f) and os.path.splitext(f)[1].lower() in ['.jpg', '.jpeg', '.png']
}

TRAIN_JSON = "/kaggle/input/datasets/sepehrhonarparvar/uit-adrone-yolo/train.json"
TEST_JSON = "/kaggle/input/datasets/sepehrhonarparvar/uit-adrone-yolo/test.json"

train_x_cpu, train_y_cpu = preload_crops_to_tensors(TRAIN_JSON, image_path_map, max_per_class=5000)
test_x_cpu, test_y_cpu = preload_crops_to_tensors(TEST_JSON, image_path_map, max_per_class=1000)

# ==============================================================================
# 2. KIẾN TRÚC MẠNG CSNN TÍCH HỢP BATCHNORM2D (TƯƠNG THÍCH CUDA)
# ==============================================================================
spike_grad = surrogate.fast_sigmoid(slope=25)

class HighAccCSNN(nn.Module):
    def __init__(self, num_classes=8, beta=0.5):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 24, kernel_size=3, stride=2, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(24)
        self.lif1 = snn.Leaky(beta=beta, spike_grad=spike_grad)

        self.conv2 = nn.Conv2d(24, 48, kernel_size=3, stride=2, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(48)
        self.lif2 = snn.Leaky(beta=beta, spike_grad=spike_grad)

        self.conv3 = nn.Conv2d(48, 64, kernel_size=3, stride=2, padding=1, bias=False)
        self.bn3 = nn.BatchNorm2d(64)
        self.lif3 = snn.Leaky(beta=beta, spike_grad=spike_grad)

        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(64 * 8 * 8, 128)
        self.lif4 = snn.Leaky(beta=beta, spike_grad=spike_grad)

        self.fc2 = nn.Linear(128, num_classes)
        self.lif5 = snn.Leaky(beta=beta, spike_grad=spike_grad)

    def forward(self, x, num_steps=8):
        B = x.size(0)
        dev = x.device
        
        # Khởi tạo tường minh điện thế màng đúng kích thước và đúng GPU
        mem1 = torch.zeros((B, 24, 32, 32), device=dev)
        mem2 = torch.zeros((B, 48, 16, 16), device=dev)
        mem3 = torch.zeros((B, 64, 8, 8), device=dev)
        mem4 = torch.zeros((B, 128), device=dev)
        mem5 = torch.zeros((B, 8), device=dev)
        
        spk5_sum = torch.zeros((B, 8), device=dev)

        # Tính trước Conv1 + BN1 ngoài vòng lặp thời gian
        cur1 = self.bn1(self.conv1(x))
        
        for _ in range(num_steps):
            spk1, mem1 = self.lif1(cur1, mem1)
            spk2, mem2 = self.lif2(self.bn2(self.conv2(spk1)), mem2)
            spk3, mem3 = self.lif3(self.bn3(self.conv3(spk2)), mem3)
            
            flat = self.dropout(spk3.view(B, -1))
            spk4, mem4 = self.lif4(self.fc1(flat), mem4)
            spk5, mem5 = self.lif5(self.fc2(spk4), mem5)
            
            spk5_sum += spk5
            
        return spk5_sum

# ==============================================================================
# 3. ĐẨY TOÀN BỘ DỮ LIỆU LÊN VRAM GPU & HUẤN LUYỆN TỐC ĐỘ CAO
# ==============================================================================
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = HighAccCSNN(num_classes=8).to(device)

# Nạp toàn bộ dữ liệu (~200MB) thẳng vào VRAM của GPU
train_x_gpu = train_x_cpu.to(device)
train_y_gpu = train_y_cpu.to(device)
test_x_gpu = test_x_cpu.to(device)
test_y_gpu = test_y_cpu.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=25)
criterion = nn.CrossEntropyLoss()

BATCH_SIZE = 256
EPOCHS = 120
num_train = train_x_gpu.size(0)
num_test = test_x_gpu.size(0)
best_acc = 0.0

print(f"\nBắt đầu huấn luyện trên {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}...")

for epoch in range(EPOCHS):
    t_ep = time.time()
    model.train()
    
    # Xáo trộn index ngẫu nhiên ngay trên GPU
    perm = torch.randperm(num_train, device=device)
    
    for i in range(0, num_train, BATCH_SIZE):
        idx = perm[i : i + BATCH_SIZE]
        x = train_x_gpu[idx].float() / 255.0
        
        # Data Augmentation siêu tốc trực tiếp trên VRAM GPU
        if random.random() < 0.5:
            x = torch.flip(x, dims=[3])  # Lật ngang ngẫu nhiên
        if random.random() < 0.5:
            scale = random.uniform(0.85, 1.15)
            shift = random.uniform(-0.08, 0.08)
            x = torch.clamp(x * scale + shift, 0.0, 1.0)
            
        y = train_y_gpu[idx]

        optimizer.zero_grad()
        out = model(x, num_steps=8)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        
    scheduler.step()

    # Đánh giá trên tập Test
    model.eval()
    correct = 0
    with torch.no_grad():
        for i in range(0, num_test, BATCH_SIZE):
            x = test_x_gpu[i : i + BATCH_SIZE].float() / 255.0
            y = test_y_gpu[i : i + BATCH_SIZE]
            preds = model(x, num_steps=8).argmax(dim=1)
            correct += (preds == y).sum().item()

    acc = 100.0 * correct / num_test
    print(f"Epoch {epoch+1:02d}/{EPOCHS:02d} ({time.time()-t_ep:.2f}s) | Test Acc: {acc:.2f}%")
    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), "best_fused_csnn.pth")

print(f"\n[HOÀN TẤT] Độ chính xác cao nhất trên tập Test: {best_acc:.2f}%")

# ==============================================================================
# 4. GỘP TRỌNG SỐ CONV2D + BATCHNORM & XUẤT FILE ZIP CHO RASPBERRY PI 4
# ==============================================================================
def fuse_conv_bn(conv, bn):
    """Gộp toán học BatchNorm2d vào Conv2d để chạy mượt trên Pi 4 mà không tốn thêm phép tính"""
    w = conv.weight.detach().cpu()
    mean = bn.running_mean.detach().cpu()
    var = bn.running_var.detach().cpu()
    gamma = bn.weight.detach().cpu()
    beta = bn.bias.detach().cpu()
    eps = bn.eps

    std = torch.sqrt(var + eps)
    scale = (gamma / std).reshape(-1, 1, 1, 1)
    fused_w = (w * scale).numpy().astype(np.float32)
    fused_b = (beta - gamma * mean / std).numpy().astype(np.float32)
    return fused_w, fused_b

model.load_state_dict(torch.load("best_fused_csnn.pth", map_location="cpu"))
model.to("cpu").eval()

c1_w, c1_b = fuse_conv_bn(model.conv1, model.bn1)
c2_w, c2_b = fuse_conv_bn(model.conv2, model.bn2)
c3_w, c3_b = fuse_conv_bn(model.conv3, model.bn3)

exports = {
    "conv1_w.npy": c1_w, "conv1_b.npy": c1_b,
    "conv2_w.npy": c2_w, "conv2_b.npy": c2_b,
    "conv3_w.npy": c3_w, "conv3_b.npy": c3_b,
    "fc1_w.npy": model.fc1.weight.detach().numpy().astype(np.float32),
    "fc1_b.npy": model.fc1.bias.detach().numpy().astype(np.float32),
    "fc2_w.npy": model.fc2.weight.detach().numpy().astype(np.float32),
    "fc2_b.npy": model.fc2.bias.detach().numpy().astype(np.float32),
}

with zipfile.ZipFile("uit_adrone_csnn_v2.zip", "w") as zf:
    for fname, arr in exports.items():
        np.save(fname, arr)
        zf.write(fname)

print("[OK] Đã xuất và nén thành công: /kaggle/working/uit_adrone_csnn_v2.zip")

[OK] Nạp 33941 mẫu trong 67.5s (139.0 MB)
[OK] Nạp 8000 mẫu trong 31.5s (32.8 MB)

Bắt đầu huấn luyện trên Tesla T4...
Epoch 01/120 (19.37s) | Test Acc: 18.76%
Epoch 02/120 (15.29s) | Test Acc: 22.65%
Epoch 03/120 (15.40s) | Test Acc: 20.75%
Epoch 04/120 (15.55s) | Test Acc: 23.20%
Epoch 05/120 (15.70s) | Test Acc: 18.66%
Epoch 06/120 (15.87s) | Test Acc: 20.86%
Epoch 07/120 (16.05s) | Test Acc: 22.50%
Epoch 08/120 (16.27s) | Test Acc: 23.04%
Epoch 09/120 (16.45s) | Test Acc: 21.80%
Epoch 10/120 (16.41s) | Test Acc: 21.61%
Epoch 11/120 (16.23s) | Test Acc: 18.96%
Epoch 12/120 (16.19s) | Test Acc: 23.46%
Epoch 13/120 (16.24s) | Test Acc: 22.84%
Epoch 14/120 (16.33s) | Test Acc: 19.62%
Epoch 15/120 (16.31s) | Test Acc: 25.56%
Epoch 16/120 (16.28s) | Test Acc: 27.09%
Epoch 17/120 (16.26s) | Test Acc: 27.61%
Epoch 18/120 (16.26s) | Test Acc: 26.62%
Epoch 19/120 (16.28s) | Test Acc: 28.52%
Epoch 20/120 (16.29s) | Test Acc: 30.30%
Epoch 21/120 (16.31s) | Test Acc: 29.05%
Epoch 22/120 (16.32s